In [2]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache 
import datetime


In [ ]:
from typing import Annotated
import operator

def update_function(old, new):
    print("old ->", old)
    print("new ->", new)
    return old + new

class State(TypedDict):
    messages: Annotated[list[str], update_function] 
    # messages: Annotated[list[str], operator.add]

class ConditionState(TypedDict):
    seed: int

class CacheState(TypedDict):
    time: str

class PrivateState(TypedDict):
    a: int
    b: int

class MegaPrivateState(TypedDict):
    secret: bool

class InputState(TypedDict):
    hello: str

class OutputState(TypedDict):
    bye: str

graph_builder = StateGraph(PrivateState, input_schema=InputState, output_schema=OutputState)

In [4]:


def node_zero(state: State):
    return {
        "messages": ["hello world!"]
    }

def node_one(state: CachePolicy):
    print("node_one", state)
    return {
        "time": f"{datetime.now()}"
    }

def node_two(state: PrivateState) -> PrivateState:
    print("node_two", state)
    return {
        "a": 1
    }

def node_three(state: PrivateState) -> PrivateState:
    print("node_three", state)
    return {
       "b": 1
    }

def node_four(state: PrivateState) -> OutputState:
    print("node_four", state)
    return {
        "bye": "world"
    }

def node_five(state: OutputState):
    print("node_five", state)
    return {
        "secret": True
    }

def node_six(state: MegaPrivateState):
    print("node_six", state)
    # return {
    #     "secret": True
    # }

In [ ]:
from typing import Literal


graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two, cache_policy=CachePolicy(ttl=20))
graph_builder.add_node("node_three", node_three)
graph_builder.add_node("node_four", node_four)
graph_builder.add_node("node_five", node_five)
graph_builder.add_node("node_six", node_six)

# def decide_path(state: ConditionState) -> Literal["node_three", "node_four"]:
#     if state["seed"] % 2 == 0: 
#         return "node_three"
#     else:
#         return "node_four"

def decide_path(state: ConditionState):
    return state["seed"] % 2 == 0

graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
# graph_builder.add_conditional_edges("node_two", decide_path)
graph_builder.add_conditional_edges("node_two", decide_path, {True: 'node_three', False: 'node_four', 'hello': END})
graph_builder.add_edge("node_three", "node_four")
graph_builder.add_edge("node_four", "node_five")
graph_builder.add_edge("node_five", "node_six")
graph_builder.add_edge("node_six", END)

In [6]:
graph = graph_builder.compile(cache=InMemoryCache())

graph

result = graph.invoke({
    'hello': 'world'
})

print(result)

node_one CachePolicy(key_func=<function default_cache_key at 0x10ff7ee80>, ttl=None)


AttributeError: module 'datetime' has no attribute 'now'

In [ ]:
print(graph.get_graph().draw_ascii())

NameError: name 'graph' is not defined

In [7]:
import time
print(graph.invoke({}))
time.sleep(5)
print(graph.invoke({}))
time.sleep(5)
print(graph.invoke({}))
time.sleep(5)
print(graph.invoke({}))
time.sleep(5)
print(graph.invoke({}))
time.sleep(5)
print(graph.invoke({}))
time.sleep(5)


node_one CachePolicy(key_func=<function default_cache_key at 0x10ff7ee80>, ttl=None)


AttributeError: module 'datetime' has no attribute 'now'